# 02 - UDA-Hub core database

Creates `data/core/udahub.db` with the six required tables (Account, User,
Ticket, TicketMetadata, TicketMessage, Knowledge) plus four tables the agent
system adds: `customer_memories` (long-term memory), `knowledge_embeddings`
(RAG vector cache), `agent_events` (structured decision log) and
`support_actions` (refund requests awaiting a support lead).

The knowledge base grows from 4 to 29 CultPass articles; they are generated
by `scripts/build_articles.py` into `data/external/cultpass_articles.jsonl`.

In [1]:
import json
import uuid
from sqlalchemy import create_engine

from utils import reset_db, get_session, model_to_dict
from data.models import udahub

# Udahub Application

## Core Database

**Init DB**

In [2]:
udahub_db = "data/core/udahub.db"

In [3]:
reset_db(udahub_db)

✅ Removed existing data/core/udahub.db
2026-09-27 02:20:40,561 INFO sqlalchemy.engine.Engine BEGIN (implicit)


2026-09-27 02:20:40,564 INFO sqlalchemy.engine.Engine COMMIT


✅ Recreated data/core/udahub.db with fresh schema


In [4]:
engine = create_engine(f"sqlite:///{udahub_db}", echo=False)
udahub.Base.metadata.create_all(bind=engine)

**Account**

In [5]:
account_id = "cultpass"
account_name = "CultPass Card"

In [6]:
with get_session(engine) as session:
    account = udahub.Account(
        account_id=account_id,
        account_name=account_name,
    )
    session.add(account)

## Integrations

**Knowledge Base**

In [7]:

# The 25 additional articles live in data/external/cultpass_articles.jsonl,
# generated by scripts/build_articles.py (the first 4 are CultPass's originals).
# Categories covered: billing, subscription, reservations, technical, account,
# security, privacy, safety, general and the escalation policy itself.


In [8]:
cultpass_articles = []

with open('data/external/cultpass_articles.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_articles.append(json.loads(line))

In [9]:

from collections import Counter

print(len(cultpass_articles), "articles")
for a in cultpass_articles:
    print(f"- {a['title']:<60} [{a['tags'].split(',')[0]}]")
print(Counter(a["tags"].split(",")[0].strip() for a in cultpass_articles))


29 articles
- How to Reserve a Spot for an Event                           [reservation]
- What's Included in a CultPass Subscription                   [subscription]
- How to Cancel or Pause a Subscription                        [cancelation]
- How to Handle Login Issues?                                  [login]
- Payment Failed or Card Declined                              [billing]
- Double Charge or Unrecognized Charge                         [billing]
- Refund Policy and Refund Requests                            [refund]
- Updating Your Payment Method or Billing Date                 [billing]
- Gift Cards and Corporate Plans                               [gift card]
- CultPass Plans: Basic vs Premium                             [subscription]
- Upgrading or Downgrading Your Plan                           [subscription]
- How the Monthly Experience Quota Works                       [subscription]
- Reactivating a Paused or Cancelled Subscription              [subscription]
- Cance

In [10]:
if len(cultpass_articles) < 14:
    raise AssertionError("You should load the articles with at least 14 records")

In [11]:
with get_session(engine) as session:
    kb = []
    for article in cultpass_articles:
        knowledge = udahub.Knowledge(
            article_id=str(uuid.uuid4()),
            account_id=account_id,
            title=article["title"],
            content=article["content"],
            tags=article["tags"]
        )
        kb.append(knowledge)
    session.add_all(kb) 
    

**Ticket**

In [12]:
cultpass_users = []

with open('data/external/cultpass_users.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        cultpass_users.append(json.loads(line))

In [13]:
ticket_info = {
    "status": "open",
    "content": "I can't log in to my Cultpass account.",
    "owner_id": cultpass_users[0]["id"],
    "owner_name": cultpass_users[0]["name"],
    "role": "user",
    "channel": "chat",
    "tags": "login, access",
}

In [14]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()

    if not user:
        user = udahub.User(
            user_id=str(uuid.uuid4()),
            account_id=account_id,
            external_user_id=ticket_info["owner_id"],
            user_name=ticket_info["owner_name"],
        )
    
    ticket = udahub.Ticket(
        ticket_id=str(uuid.uuid4()),
        account_id=account_id,
        user_id=user.user_id,
        channel=ticket_info["channel"],
    )
    metadata = udahub.TicketMetadata(
        ticket_id=ticket.ticket_id,
        status=ticket_info["status"],
        main_issue_type=None,
        tags=ticket_info["tags"],
    )

    first_message = udahub.TicketMessage(
        message_id=str(uuid.uuid4()),
        ticket_id=ticket.ticket_id,
        role=ticket_info["role"],
        content=ticket_info["content"],
    )

    session.add_all([user, ticket, metadata, first_message])


**Ticket history**

Earlier, already-closed tickets for two customers, so the agents can show
"returning customer" behaviour (history is loaded by the intake agent).

In [15]:
from datetime import datetime, timedelta

historical_tickets = [
    {
        "owner_id": "f556c0", "channel": "email", "status": "resolved", "issue_type": "technical_issue",
        "tags": "qr code, check-in", "days_ago": 20,
        "messages": [
            ("user", "The QR code wouldn't scan at the museum entrance yesterday."),
            ("ai", "Sorry about that! Raise your screen brightness and open the ticket from 'My Reservations' while online; the code refreshes every minute so screenshots won't scan."),
            ("user", "That worked today, thanks."),
        ],
    },
    {
        "owner_id": "88382b", "channel": "chat", "status": "resolved", "issue_type": "reservation_booking",
        "tags": "premium, fees", "days_ago": 35,
        "messages": [
            ("user", "Do I pay extra for premium events on my plan?"),
            ("ai", "On Premium there's no extra fee for premium experiences; each booking still uses one monthly credit."),
        ],
    },
]

with get_session(engine) as session:
    for info in historical_tickets:
        owner = next(u for u in cultpass_users if u["id"] == info["owner_id"])
        user = session.query(udahub.User).filter_by(account_id=account_id, external_user_id=owner["id"]).first()
        if not user:
            user = udahub.User(user_id=str(uuid.uuid4()), account_id=account_id,
                               external_user_id=owner["id"], user_name=owner["name"])
            session.add(user)
            session.flush()
        created = datetime.now() - timedelta(days=info["days_ago"])
        ticket = udahub.Ticket(ticket_id=str(uuid.uuid4()), account_id=account_id, user_id=user.user_id,
                               channel=info["channel"], created_at=created)
        session.add(ticket)
        session.add(udahub.TicketMetadata(ticket_id=ticket.ticket_id, status=info["status"],
                                          main_issue_type=info["issue_type"], tags=info["tags"]))
        for n, (role, content) in enumerate(info["messages"]):
            session.add(udahub.TicketMessage(message_id=str(uuid.uuid4()), ticket_id=ticket.ticket_id,
                                             role=role, content=content,
                                             created_at=created + timedelta(minutes=5 * n)))

# Tests

In [16]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    print(account)

<Account(account_id='cultpass', account_name='CultPass Card')>


In [17]:
with get_session(engine) as session:
    account = session.query(udahub.Account).filter_by(
        account_id=account_id
    ).first()
    for article in account.knowledge_articles:
        print(article)

<Knowledge(article_id='66d03069-a186-4698-b077-d63762aa5c4c', title='How to Reserve a Spot for an Event')>
<Knowledge(article_id='938f535a-03e3-4cad-8e2a-8dd8e093208d', title='What's Included in a CultPass Subscription')>
<Knowledge(article_id='7336b04e-a9e4-4d35-ab35-2984df42ee91', title='How to Cancel or Pause a Subscription')>
<Knowledge(article_id='ad684f86-ec22-4de8-89d3-f72e3305384e', title='How to Handle Login Issues?')>
<Knowledge(article_id='5f9d3985-3821-4680-b98d-bcdfaccbddb9', title='Payment Failed or Card Declined')>
<Knowledge(article_id='ea37e8fa-9d37-443e-855c-8ea6c6b63ca9', title='Double Charge or Unrecognized Charge')>
<Knowledge(article_id='b633e00c-07ee-4ee4-b5f6-a8ee821ed954', title='Refund Policy and Refund Requests')>
<Knowledge(article_id='0fe3838d-4c47-4d70-9bf4-4e570183b458', title='Updating Your Payment Method or Billing Date')>
<Knowledge(article_id='35c8137a-7b8c-4fd0-bb2f-ff1fe0bd6d06', title='Gift Cards and Corporate Plans')>
<Knowledge(article_id='7f9a31

In [18]:
with get_session(engine) as session:
    users = session.query(udahub.User).all()
    for user in users:
        print(user)

<User(user_id='86483f9f-ec7d-4ac5-bb63-08e12c7a9245', user_name='Alice Kingsley', external_user_id='a4ab87')>
<User(user_id='f98e2701-5d5e-4d37-9d3c-956aa3a14109', user_name='Bob Stone', external_user_id='f556c0')>
<User(user_id='a57a79b1-38bc-4ef7-8574-366a13946525', user_name='Cathy Bloom', external_user_id='88382b')>


In [19]:
with get_session(engine) as session:
    user = session.query(udahub.User).filter_by(
        account_id=account_id,
        external_user_id=ticket_info["owner_id"],
    ).first()
    
    ticket:udahub.Ticket = user.tickets[0]
    for message in ticket.messages:
        print(message)

<TicketMessage(message_id='adaec521-44f3-4fdb-9077-bbba795fecd1', role='user', content='I can't log in to my Cultpass ...')>


In [20]:

from sqlalchemy import inspect

required = {"accounts", "users", "tickets", "ticket_metadata", "ticket_messages", "knowledge"}
tables = set(inspect(engine).get_table_names())
print("tables:", sorted(tables))
assert required <= tables, required - tables

with get_session(engine) as session:
    for model in (udahub.Account, udahub.User, udahub.Ticket, udahub.TicketMetadata,
                  udahub.TicketMessage, udahub.Knowledge):
        print(f"{model.__tablename__:<16} {session.query(model).count():>3} rows")
    assert session.query(udahub.Knowledge).count() >= 14


tables: ['accounts', 'agent_events', 'customer_memories', 'knowledge', 'knowledge_embeddings', 'support_actions', 'ticket_messages', 'ticket_metadata', 'tickets', 'users']
accounts           1 rows
users              3 rows
tickets            3 rows
ticket_metadata    3 rows
ticket_messages    6 rows
knowledge         29 rows


In [21]:
# Data retrieval demo: the knowledge articles about refunds, straight from the DB
with get_session(engine) as session:
    for k in session.query(udahub.Knowledge).filter(udahub.Knowledge.tags.contains("refund")).all():
        print(k.title, "->", k.content.splitlines()[0])

Double Charge or Unrecognized Charge -> When a user reports being charged twice or does not recognize a charge:
Refund Policy and Refund Requests -> CultPass refund policy:
Event Cancelled or Rescheduled by the Partner -> When a partner venue cancels or reschedules:
